# Tome 5, Chapter 9: The Erdős–Kac theorem (Python companion)

Exercises first, solutions at the end. We tabulate $\omega(n)$ and $\Omega(n)$ up to $10^7$, compare with the Gaussian law of Theorem 9.7, check Turán's variance (Theorem 9.2), compute the exact law of Kubilius's model, watch the moments of the integers agree with those of the model as Lemma 9.5 promises, look at where the mean of $2^{\omega(n)}$ comes from (Exercise 9.2), count the multiplication table (Exercise 9.5), and factor random $18$-digit integers. Requirements: numpy, sympy.

*Before each solution, predict what the output will look like.*

## Exercises

**P9.1 (★) The data.** Compute $\omega(n)$ for $n\le10^7$ with a sieve. Compare the mean and variance with $\log\log x$, and the proportions $P(Y\le a)$, $Y=\frac{\omega(n)-\log\log x}{\sqrt{\log\log x}}$, with $\Phi(a)$.

**P9.2 (★) Turán's variance.** Compute $\frac1x\sum_{n\le x}(\omega(n)-\log\log x)^2$ for $x=10^4,\dots,10^7$. Is it $O(\log\log x)$? Is it $\sim\log\log x$?

**P9.3 (★★) The model.** Compute the exact law of $\sum_{p\le z}X_p$ (independent Bernoulli variables of parameter $\frac1p$) for $z=x$, $\sqrt x$, $x^{1/3}$ with $x=10^7$, and compare with the data. Which $z$ fits best, and why does no $z$ fit well?

**P9.4 (★★) Moments.** With $z=x^{1/4}$, compare the moments of $\frac{\omega_z(n)-\mu_z}{\sigma_z}$ over $n\le10^7$ with those of the model and with the Gaussian moments (Lemmas 9.4 and 9.5).

**P9.5 (★) Mean against typical.** Where does the sum $\sum_{n\le x}2^{\omega(n)}$ come from? Compute the share of the sum carried by the $n$ with $\omega(n)\le\lambda\log\log x$ for a few $\lambda$, and compare the median and the mean of $2^{\omega(n)}$.

**P9.6 (★) Two exercises in numbers.** Count the distinct entries of the $N\times N$ multiplication table, and the proportion of $n\le10^7$ with a prime factor larger than $\sqrt{10^7}$ (Exercises 9.5 and 9.4).

**P9.7 (★★) CS application: factoring random integers.** Factor $1500$ random integers between $10^{18}$ and $2\cdot10^{18}$ with `sympy.factorint`. Compare the law of $\omega$ with P9.1, and look at the second largest prime factor, which governs the running time of Pollard's rho method.

## Solutions

### P9.1

In [ ]:
import math, time
import numpy as np

N = 10**7
omega = np.zeros(N + 1, dtype=np.int8)
Omega = np.zeros(N + 1, dtype=np.int8)
is_p = np.ones(N + 1, dtype=bool); is_p[:2] = False
for p in range(2, N + 1):
    if is_p[p]:
        if p * p <= N: is_p[p*p::p] = False
        omega[p::p] += 1
        q = p
        while q <= N:
            Omega[q::q] += 1; q *= p
primes = np.nonzero(is_p)[0]
w = omega[1:].astype(float)
L = math.log(math.log(N))
M = 0.2614972128
print(f"n <= 1e7: mean omega = {w.mean():.4f} (loglog x + M = {L + M:.4f}),  variance = {w.var():.4f} (loglog x = {L:.4f})")
print("P(omega = k), k = 0..7:", np.round(np.bincount(omega[1:], minlength=8)[:8] / N, 4))
Phi = lambda a: 0.5 * (1 + math.erf(a / math.sqrt(2)))
Y = (w - L) / math.sqrt(L)
for a in (-1, 0, 1, 2):
    print(f"P(Y <= {a:2d}) = {np.mean(Y <= a):.4f}   Phi({a}) = {Phi(a):.4f}")

The mean is right ($\log\log x+M$, as Chapter 1 predicts) but the spread is far too small: the variance is $1.10$ instead of $2.78$, and the proportions are nowhere near $\Phi$. The integers up to $10^7$ simply cannot have many prime factors: the product of the first nine primes already exceeds $2\cdot10^8$. Theorem 9.7 is a statement about $\log\log x\to\infty$, and $\log\log x$ grows so slowly that the saying goes: it tends to infinity, but has never been observed to do so.

### P9.2

In [ ]:
for k in range(4, 8):
    x = 10**k; Lx = math.log(math.log(x))
    v = np.mean((omega[1:x + 1] - Lx)**2)
    print(f"x = 1e{k}: (1/x) sum (omega - loglog x)^2 = {v:.4f},  divided by loglog x: {v / Lx:.4f}")

The ratio is far below $1$ and creeps up slowly. In fact $\frac1x\sum_{n\le x}(\omega(n)-\log\log x)^2=\log\log x+C+o(1)$ for a negative constant $C$ (about $-1.6$ judging from these numbers), so the ratio tends to $1$, at the speed of $\frac1{\log\log x}$. Turán's bound $O(\log\log x)$ is all the theorem needs.

### P9.3

In [ ]:
def model_pmf(z, kmax=20):
    pmf = np.zeros(kmax + 1); pmf[0] = 1.0
    for p in primes[primes <= z]:
        q = 1.0 / p
        pmf[1:] = pmf[1:] * (1 - q) + pmf[:-1] * q
        pmf[0] *= 1 - q
    return pmf
data = np.bincount(omega[1:], minlength=10)[:10] / N
for z in (N, int(N**0.5), int(N**(1/3))):
    pm = model_pmf(z)
    ks = np.arange(len(pm))
    mean = (ks * pm).sum(); var = (ks**2 * pm).sum() - mean**2
    print(f"model with z = {z:8d}: mean {mean:.3f}, variance {var:.3f}, P(k) for k = 0..6: {np.round(pm[:7], 3)}")
print(f"data                    : mean {w.mean():.3f}, variance {w.var():.3f}, P(k) for k = 0..6: {np.round(data[:7], 3)}")

With $z=x$ the model has the right mean but almost two and a half times the variance: it lets an integer below $10^7$ be divisible by several primes near $10^7$ at once, which is impossible. Cutting at $z=\sqrt x$ or $x^{1/3}$ removes that, but also removes the large prime factors that most integers do have, and the mean drops. This is the same failure as the $2e^{-\gamma}$ of Chapter 1 and the $\log2$ of Exercise 9.4: the model is exact for small primes and wrong for large ones. The proof of the theorem gets around it by using the model only for $p\le x^{1/\beta}$ and showing that the larger primes shift $\omega$ by at most $\beta=o(\sqrt{\log\log x})$.

### P9.4

In [ ]:
z = int(N**0.25)
small = primes[primes <= z]
wz = np.zeros(N + 1, dtype=np.int8)
for p in small: wz[p::p] += 1
mu = float(np.sum(1.0 / small)); sig = math.sqrt(float(np.sum((1.0 / small) * (1 - 1.0 / small))))
Yz = (wz[1:] - mu) / sig
pm = model_pmf(z)
ks = np.arange(len(pm))
print(f"z = {z}: {len(small)} primes, mu_z = {mu:.4f}, sigma_z = {sig:.4f}")
print(" k   E_N[Y^k] (integers)   E[(S/sigma)^k] (model)   Gaussian m_k")
for k in range(1, 7):
    model = float(np.sum(pm * ((ks - mu) / sig)**k))
    gauss = 0 if k % 2 else math.prod(range(1, k, 2))
    print(f" {k}   {np.mean(Yz**k):18.5f}   {model:20.5f}   {gauss:10d}")

Here the agreement between integers and model is excellent, to three or four digits, exactly as Lemma 9.5 says: with only $16$ primes up to $z=56$, the integers up to $10^7$ really are independent coin tosses for these primes. The distance to the Gaussian moments is the model's own problem: with $\sigma_z=1.11$ the third moment is still $0.5$ instead of $0$ (compare Exercise 9.3). The theorem lets $z$ and $\sigma_z$ grow, and then both columns tend to the last one.

### P9.5

In [ ]:
n = np.arange(1, N + 1)
two = 2.0**omega[1:]
tot = two.sum()
print(f"sum of 2^omega(n), n <= 1e7: {tot:.4e};  (6/pi^2) x log x = {6/math.pi**2 * N * math.log(N):.4e}")
for lam in (1.0, 1.4, 2.0):
    sel = omega[1:] <= lam * L
    print(f"n with omega(n) <= {lam} loglog x: proportion of integers {sel.mean():.4f}, share of the sum {two[sel].sum() / tot:.4f}")
logtau = np.zeros(N + 1)
# tau via smallest-prime-factor exponents: use Omega and omega only for bounds
print(f"median of 2^omega(n) for n <= 1e7: {np.median(two):.0f};  (log x)^(log 2) = {math.log(N)**math.log(2):.2f};  mean = {two.mean():.2f}")

The median of $2^{\omega(n)}$ is $8$, close to $(\log x)^{\log2}=6.9$; the mean is $10.6$, close to $\frac6{\pi^2}\log x$. At $10^7$ the integers with $\omega(n)\le1.4\log\log x$ still carry $38\%$ of the sum, so Exercise 9.2 is again a theorem about astronomically large $x$; but the trend is visible: $68\%$ of the integers carry only $38\%$ of the sum, and the last $0.75\%$ of the integers carry almost $5\%$.

### P9.6

In [ ]:
for Nn in (10, 100, 1000, 2000):
    a = np.arange(1, Nn + 1)
    D = len(np.unique(np.outer(a, a)))
    print(f"N = {Nn:5d}: distinct products D(N) = {D:8d},  D(N)/N^2 = {D / Nn**2:.4f}")
lp = np.zeros(N + 1, dtype=np.int64)
for p in primes: lp[p::p] = p
big = np.mean(lp[1:] > math.isqrt(N))
print(f"proportion of n <= 1e7 with a prime factor > sqrt(1e7): {big:.4f};  log 2 = {math.log(2):.4f};  independent model: 1/2")

$D(N)/N^2$ decreases, but so slowly that one could believe it tends to a positive limit; Exercise 9.5 proves it tends to $0$, and Ford's theorem says it is about $(\log N)^{-0.086}$, which at $N=2000$ is $0.84$ before the other factors: no computer will see it reach $0.1$. The proportion of integers with a large prime factor is $0.66$, on its way to $\log2=0.693$ (the error is of order $\frac1{\log x}$), clearly away from the $\frac12$ of the independent model.

### P9.7

In [ ]:
import random
from sympy import factorint
random.seed(2026)
samples = [random.randrange(10**18, 2 * 10**18) for _ in range(1500)]
t0 = time.perf_counter()
facts = [factorint(m) for m in samples]
t1 = time.perf_counter()
om18 = np.array([len(f) for f in facts])
L18 = math.log(math.log(1.5e18))
print(f"1500 random integers near 1e18 factored in {t1 - t0:.1f} s")
print(f"mean omega = {om18.mean():.3f} (loglog x + M = {L18 + M:.3f}), variance = {om18.var():.3f}, P(omega = k), k = 1..8: {np.round(np.bincount(om18, minlength=9)[1:9] / len(om18), 3)}")
second = []
for f in facts:
    ps = sorted(p for p, e in f.items() for _ in range(e))
    second.append(ps[-2] if len(ps) > 1 else 1)
second = np.array(second, dtype=float)
print(f"second largest prime factor: median {np.median(second):.3g}, 90% quantile {np.quantile(second, 0.9):.3g}; Pollard rho needs about sqrt of it steps")

Near $10^{18}$, $\log\log x=3.73$, and the mean of $\omega$ matches $\log\log x+M$; the variance, $2.0$, is still below $\log\log x$ but closer than at $10^7$. Factoring is fast because most integers have one or two large prime factors and many small ones: trial division and Pollard's rho strip the small ones, and the cost of rho is about the square root of the second largest prime factor, which is typically tiny compared with $10^{18}$. The hard case for factoring, the product of two primes of equal size used in RSA, is exactly what the Erdős–Kac law says almost never happens by chance.